In [1]:
from pathlib import Path
import gc

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse


# =========================================================
# 0. paths and settings
# =========================================================
PROJECT_DIR = Path(".")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROC_DIR = PROJECT_DIR / "data" / "processed"
META_DIR = PROC_DIR / "metadata"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

ADATA1_PATH = RAW_DIR / "dataset1.h5ad"
ADATA2_PATH = RAW_DIR / "dataset2.h5ad"

# label harmonization
LABEL_MAP = {
    "pancreatic A cell": "alpha",
    "type B pancreatic cell": "beta",
    "pancreatic D cell": "delta",
    "pancreatic PP cell": "pp",
    "PP cell": "pp",
}

KEEP_LABELS = ["alpha", "beta", "delta", "pp"]

ADATA2_SPLITS = {
    "control": "Control",
    "aab": "AAB",
    "t1d": "T1D",
}


# =========================================================
# 1. helper functions
# =========================================================
def ensure_csr_matrix(x):
    """Ensure matrix is CSR sparse matrix."""
    if sparse.issparse(x):
        return x.tocsr()
    return sparse.csr_matrix(x)


def extract_raw_as_anndata(adata):
    """
    Use adata.raw as the count matrix if available.
    Return a new AnnData with:
    - X = raw counts
    - obs copied from current object
    - var copied from raw.var
    """
    if adata.raw is not None:
        new_adata = ad.AnnData(
            X=ensure_csr_matrix(adata.raw.X.copy()),
            obs=adata.obs.copy(),
            var=adata.raw.var.copy(),
            uns=adata.uns.copy(),
        )
    else:
        new_adata = adata.copy()
        new_adata.X = ensure_csr_matrix(new_adata.X)

    # gene symbol column
    if "feature_name" in new_adata.var.columns:
        new_adata.var["gene_symbol"] = new_adata.var["feature_name"].astype(str)
    else:
        new_adata.var["gene_symbol"] = new_adata.var_names.astype(str)

    # make sure var_names are unique
    new_adata.var_names_make_unique()

    return new_adata


def add_harmonized_labels(adata):
    """Add harmonized endocrine labels."""
    adata = adata.copy()

    if "cell_type" not in adata.obs.columns:
        raise ValueError("`cell_type` column not found in adata.obs")

    adata.obs["cell_type_original"] = adata.obs["cell_type"].astype(str)
    adata.obs["label"] = adata.obs["cell_type_original"].map(LABEL_MAP)

    adata.obs["label"] = pd.Categorical(
        adata.obs["label"],
        categories=KEEP_LABELS,
        ordered=False
    )

    return adata


def subset_endocrine(adata):
    """Keep only mapped endocrine classes."""
    mask = adata.obs["label"].notna()
    return adata[mask].copy()


def subset_adata2_by_disease_state(adata2):
    """Split adata2 into Control / AAB / T1D."""
    if "disease_state" not in adata2.obs.columns:
        raise ValueError("`disease_state` column not found in adata2.obs")

    split_dict = {}
    for split_name, split_value in ADATA2_SPLITS.items():
        split_dict[split_name] = adata2[
            adata2.obs["disease_state"].astype(str) == split_value
        ].copy()

    return split_dict


def intersect_genes(adatas):
    """Get shared var_names across all input AnnData objects, preserving order of the first."""
    if len(adatas) == 0:
        raise ValueError("`adatas` is empty")

    shared = adatas[0].var_names
    for a in adatas[1:]:
        shared = shared.intersection(a.var_names, sort=False)
    return pd.Index(shared)


def align_to_shared_genes(adata, shared_genes):
    """Subset and reorder genes to a common shared gene index."""
    return adata[:, shared_genes].copy()


def add_basic_qc_metrics(adata):
    """Compute QC metrics on current X."""
    sc.pp.calculate_qc_metrics(
        adata,
        inplace=True,
        log1p=False,
        percent_top=None
    )
    return adata


def filter_empty_cells(adata):
    """Remove cells with zero total counts, if any."""
    sc.pp.filter_cells(adata, min_counts=1)
    return adata


def make_log1p_copy_from_counts(adata_counts):
    """
    Create a normalized + log1p copy.
    Save raw counts into layers['counts'].
    """
    adata_norm = adata_counts.copy()
    adata_norm.layers["counts"] = adata_norm.X.copy()

    sc.pp.normalize_total(adata_norm, target_sum=1e4)
    sc.pp.log1p(adata_norm)

    adata_norm.uns["preprocessing_note"] = (
        "X contains log1p(normalized counts, target_sum=1e4); "
        "raw counts are stored in layers['counts']."
    )
    return adata_norm


def save_selected_obs(adata, out_csv):
    """Save key metadata columns from obs."""
    preferred_cols = [
        "label",
        "cell_type_original",
        "cell_type",
        "donor_id",
        "sample",
        "disease",
        "disease_state",
        "sex",
        "assay",
        "tissue",
        "dataset_name",
        "dataset_role",
        "cohort",
    ]
    cols = [c for c in preferred_cols if c in adata.obs.columns]
    adata.obs[cols].to_csv(out_csv, index=True)


def dataset_summary_row(name, adata):
    row = {
        "dataset_name": name,
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
    }
    if "label" in adata.obs.columns:
        vc = adata.obs["label"].value_counts().reindex(KEEP_LABELS, fill_value=0)
        for lab in KEEP_LABELS:
            row[f"n_{lab}"] = int(vc.get(lab, 0))
    return row


# =========================================================
# 2. load raw h5ad files
# =========================================================
print("Loading datasets...")
adata1 = sc.read_h5ad(ADATA1_PATH)
adata2 = sc.read_h5ad(ADATA2_PATH)

print("\nLoaded:")
print("adata1:", adata1)
print("adata2:", adata2)

# Save initial summary
initial_summary = pd.DataFrame([
    {
        "dataset": "adata1_original",
        "n_cells": int(adata1.n_obs),
        "n_genes": int(adata1.n_vars),
        "has_raw": adata1.raw is not None,
    },
    {
        "dataset": "adata2_original",
        "n_cells": int(adata2.n_obs),
        "n_genes": int(adata2.n_vars),
        "has_raw": adata2.raw is not None,
    },
])
initial_summary.to_csv(META_DIR / "initial_dataset_summary.csv", index=False)

# Save original cell type frequencies
adata1.obs["cell_type"].value_counts().to_csv(META_DIR / "adata1_original_cell_type_counts.csv")
adata2.obs["cell_type"].value_counts().to_csv(META_DIR / "adata2_original_cell_type_counts.csv")

# Save label map
pd.DataFrame(
    [{"original_cell_type": k, "harmonized_label": v} for k, v in LABEL_MAP.items()]
).to_csv(META_DIR / "label_mapping.csv", index=False)


# =========================================================
# 3. harmonize labels
# =========================================================
print("\nHarmonizing labels...")
adata1 = add_harmonized_labels(adata1)
adata2 = add_harmonized_labels(adata2)

# Keep only endocrine labels
adata1_endo = subset_endocrine(adata1)
adata2_endo = subset_endocrine(adata2)

print("adata1_endo:", adata1_endo)
print("adata2_endo:", adata2_endo)

print("\nadata1 endocrine label counts:")
print(adata1_endo.obs["label"].value_counts())

print("\nadata2 endocrine label counts:")
print(adata2_endo.obs["label"].value_counts())


# =========================================================
# 4. split adata2 by disease_state
# =========================================================
print("\nSplitting adata2 by disease_state...")
adata2_splits = subset_adata2_by_disease_state(adata2_endo)

adata2_control_endo = adata2_splits["control"]
adata2_aab_endo = adata2_splits["aab"]
adata2_t1d_endo = adata2_splits["t1d"]

print("adata2_control_endo:", adata2_control_endo)
print("adata2_aab_endo:", adata2_aab_endo)
print("adata2_t1d_endo:", adata2_t1d_endo)

# Save cell counts after split
split_counts = pd.DataFrame([
    dataset_summary_row("adata1_endo", adata1_endo),
    dataset_summary_row("adata2_control_endo", adata2_control_endo),
    dataset_summary_row("adata2_aab_endo", adata2_aab_endo),
    dataset_summary_row("adata2_t1d_endo", adata2_t1d_endo),
])
split_counts.to_csv(META_DIR / "endocrine_split_summary.csv", index=False)


# =========================================================
# 5. extract counts from .raw
# =========================================================
print("\nExtracting count matrices from .raw...")
adata1_counts = extract_raw_as_anndata(adata1_endo)
adata2_control_counts = extract_raw_as_anndata(adata2_control_endo)
adata2_aab_counts = extract_raw_as_anndata(adata2_aab_endo)
adata2_t1d_counts = extract_raw_as_anndata(adata2_t1d_endo)

# annotate dataset role / cohort
adata1_counts.obs["dataset_name"] = "adata1_reference"
adata1_counts.obs["dataset_role"] = "reference"
adata1_counts.obs["cohort"] = "reference"

adata2_control_counts.obs["dataset_name"] = "adata2_control"
adata2_control_counts.obs["dataset_role"] = "query"
adata2_control_counts.obs["cohort"] = "control"

adata2_aab_counts.obs["dataset_name"] = "adata2_aab"
adata2_aab_counts.obs["dataset_role"] = "query"
adata2_aab_counts.obs["cohort"] = "aab"

adata2_t1d_counts.obs["dataset_name"] = "adata2_t1d"
adata2_t1d_counts.obs["dataset_role"] = "query"
adata2_t1d_counts.obs["cohort"] = "t1d"


# =========================================================
# 6. intersect genes
# =========================================================
print("\nFinding shared genes across all processed datasets...")
shared_genes = intersect_genes([
    adata1_counts,
    adata2_control_counts,
    adata2_aab_counts,
    adata2_t1d_counts,
])

print(f"Number of shared genes: {len(shared_genes)}")

# save shared gene table
shared_gene_table = pd.DataFrame({
    "ensembl_id": shared_genes,
    "gene_symbol": adata1_counts.var.loc[shared_genes, "gene_symbol"].astype(str).values
})
shared_gene_table.to_csv(META_DIR / "shared_genes.csv", index=False)


# =========================================================
# 7. align all datasets to shared genes
# =========================================================
print("\nAligning datasets to shared genes...")
adata1_ref = align_to_shared_genes(adata1_counts, shared_genes)
adata2_query_control = align_to_shared_genes(adata2_control_counts, shared_genes)
adata2_query_aab = align_to_shared_genes(adata2_aab_counts, shared_genes)
adata2_query_t1d = align_to_shared_genes(adata2_t1d_counts, shared_genes)

# free memory
del adata1_counts, adata2_control_counts, adata2_aab_counts, adata2_t1d_counts
gc.collect()


# =========================================================
# 8. minimal QC on aligned counts data
# =========================================================
print("\nRunning minimal QC...")
aligned_counts = {
    "pancreas_ref_counts_endocrine4_sharedgenes": adata1_ref,
    "pancreas_query_control_counts_endocrine4_sharedgenes": adata2_query_control,
    "pancreas_query_aab_counts_endocrine4_sharedgenes": adata2_query_aab,
    "pancreas_query_t1d_counts_endocrine4_sharedgenes": adata2_query_t1d,
}

summary_rows = []

for name, a in aligned_counts.items():
    filter_empty_cells(a)
    add_basic_qc_metrics(a)
    summary_rows.append(dataset_summary_row(name, a))

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(META_DIR / "processed_dataset_summary.csv", index=False)

# save cell counts per dataset
cell_count_rows = []
for name, a in aligned_counts.items():
    vc = a.obs["label"].value_counts().reindex(KEEP_LABELS, fill_value=0)
    for lab in KEEP_LABELS:
        cell_count_rows.append({
            "dataset_name": name,
            "label": lab,
            "n_cells": int(vc[lab]),
        })
pd.DataFrame(cell_count_rows).to_csv(META_DIR / "processed_label_counts.csv", index=False)


# =========================================================
# 9. save aligned raw counts datasets
# =========================================================
print("\nSaving aligned raw count datasets...")
for name, a in aligned_counts.items():
    out_h5ad = PROC_DIR / f"{name}.h5ad"
    a.write_h5ad(out_h5ad, compression="gzip")

    out_obs_csv = META_DIR / f"{name}_obs.csv"
    save_selected_obs(a, out_obs_csv)

print("Raw-count aligned datasets saved.")


# =========================================================
# 10. create and save normalized log1p versions
# =========================================================
print("\nCreating normalized log1p versions...")
for name, a in aligned_counts.items():
    norm_name = name.replace("_counts_", "_log1p_")
    a_norm = make_log1p_copy_from_counts(a)

    out_h5ad = PROC_DIR / f"{norm_name}.h5ad"
    a_norm.write_h5ad(out_h5ad, compression="gzip")

    out_obs_csv = META_DIR / f"{norm_name}_obs.csv"
    save_selected_obs(a_norm, out_obs_csv)

    del a_norm
    gc.collect()

print("Normalized log1p datasets saved.")


# =========================================================
# 11. final print
# =========================================================
print("\nDone.")
print("Saved files in:", PROC_DIR.resolve())
print("Metadata files in:", META_DIR.resolve())


Loading datasets...

Loaded:
adata1: AnnData object with n_obs × n_vars = 26474 × 30172
    obs: 'sample', 'louvain_anno_broad', 'louvain_anno_fine', 'donor_id', 'BMI', 'HbA1c', 'insulin_content', 'glucose_SI', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'n_genes', 'n_counts', 'mt_frac', 'size_factors', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'batch_condition', 'citation', 'default_embedding', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_pca', 'X_umap'
adata2: AnnData object with n_obs × n_vars = 69645 × 